# Module 1: Concepts Walkthrough

This notebook walks through the internals of ParcelFlow, a small data-driven
workflow engine. Read it alongside `LECTURE_NOTES.md`.

**Learning objectives**
1. Understand the `Parcel` (data) vs `Node` (logic) split.
2. Watch the engine discover what can run from data availability alone.
3. See execution order emerge from data dependencies, with no wiring.

In [ ]:
import sys, os
# Make the engine importable whether this notebook is launched from the
# repo root or from the education/ folder.
for candidate in ('.', '..'):
    if os.path.exists(os.path.join(candidate, 'workflow_engine.py')):
        sys.path.insert(0, os.path.abspath(candidate))
        break

from workflow_engine import WorkflowEngine
from base_node import BaseNode
print('ParcelFlow loaded.')

## 1. The unit of logic: a Node

A **Node** is a self-contained unit of work. It does not know what runs before or
after it. It declares only what data it needs (`requires`) and what it produces
(`outputs`). Here is a node that adds two numbers.

In [ ]:
class AdderNode(BaseNode):
    def __init__(self):
        # Needs parcels 'a' and 'b'; produces 'sum'.
        super().__init__('adder', requires=['a', 'b'], outputs=['sum'])

    def run(self, parcels, index=None):
        a = parcels['a'].value
        b = parcels['b'].value
        print(f'  AdderNode: {a} + {b} = {a + b}')
        return {'sum': a + b}

## 2. The engine loop

The engine has no plan. Each pass it asks every node: *are all your required
parcels present yet?* It runs the ones that are ready. Watch what happens as we
supply data incrementally.

In [ ]:
engine = WorkflowEngine()
nodes = [AdderNode()]

print('--- No data: nothing is ready ---')
engine.execute_workflow(nodes, {})

In [ ]:
print('--- Partial data: still waiting for b ---')
engine.execute_workflow(nodes, {'a': 10})

In [ ]:
print('--- Full data: the node fires ---')
result = engine.execute_workflow(nodes, {'a': 10, 'b': 5})
print('\nFinal sum:', result['sum'].value)

## 3. Implicit chaining

Now add a second node that needs the first node's output. We never tell the engine
'run Adder, then Multiplier' — the `MultiplierNode` simply requires `sum`, which
`AdderNode` produces.

In [ ]:
class MultiplierNode(BaseNode):
    def __init__(self):
        super().__init__('multiplier', requires=['sum', 'c'], outputs=['product'])

    def run(self, parcels, index=None):
        s = parcels['sum'].value
        c = parcels['c'].value
        print(f'  MultiplierNode: {s} * {c} = {s * c}')
        return {'product': s * c}

engine = WorkflowEngine()
pipeline = [AdderNode(), MultiplierNode()]
result = engine.execute_workflow(pipeline, {'a': 2, 'b': 3, 'c': 4})
print('\nFinal product:', result['product'].value)

### What happened

We never declared an order. The engine derived it:

1. `a`, `b`, `c` exist at the start.
2. `AdderNode` needs `a, b` -> ready -> runs -> produces `sum`.
3. `MultiplierNode` needs `sum, c`. `sum` did not exist until step 2, so it could
   only run afterwards.

That is data-driven execution: the schedule is a consequence of the data, not
something you wrote down.